[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-4_executed.ipynb)

---

# 40. The Port Capacity and Expansion Timing Problem
## Tier 4: Transformer-Based Capacity Planning

### Goal and scope

This notebook demonstrates how a Transformer-style sequence model can turn a history of port indicators into three probabilistic outputs: a capacity action, a technology choice, and an investment-timing category. It uses synthetic monthly data, a compact NumPy implementation, and rule-generated targets. It does not use operational data from the Port of Singapore or any other port, and its outputs are not investment advice.

The encoder in this teaching implementation has fixed, randomly initialized weights. Gradient descent trains the three output heads, not the encoder. Consequently, this notebook illustrates sequence encoding, attention, multi-task outputs, and supervised head training; it does not implement a fully trained Transformer or the AdamW training procedure sometimes used for such models. The probabilities are model scores, not calibrated confidence intervals.

### Inputs, outputs, and assumptions

- **Input:** a monthly time series with 24 operational, economic, and market features.
- **Input window:** 120 months (10 years).
- **Forecast window:** 36 months (3 years), used to construct synthetic decision targets.
- **Outputs:** probabilities over four capacity actions, three technology choices, and three timing categories.
- **Data:** 180 synthetic months generated in this notebook. The labels are generated by explicit rules from the future window; they are not independently observed decisions.

For this demonstration, feature scaling is fitted to the supplied synthetic history, and predictions are evaluated on the same set of windows used for training. There is no held-out test set, external validation, investment-cost model, or explicit optimization under operational constraints. These assumptions make the example reproducible and interpretable as a coding exercise, but they do not support claims about real-world predictive performance.

### Why use attention?

A time-series window contains ordered observations: each row represents one month, and each column represents one feature. Attention allows each month’s representation to combine information from other months in the window. Positional encoding supplies information about order because attention by itself does not distinguish the first month from the last.

For queries \(Q\), keys \(K\), and values \(V\), scaled dot-product attention is

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^{\mathsf T}}{\sqrt{d_k}}\right)V.$$

Here, \(d_k\) is the dimension of each attention head. The dot products measure how strongly positions relate; division by \(\sqrt{d_k}\) keeps their scale manageable; and softmax turns each row of scores into nonnegative weights summing to one. Multiple heads perform this calculation in separate learned subspaces. In this notebook the attention weights are fixed, so they illustrate the computation rather than learned evidence about port operations.

### Target construction and learning objective

Each future window is converted into a one-hot capacity target \(y^{(c)}\), a one-hot technology target \(y^{(g)}\), and a normalized timing target \(y^{(t)}\). A one-hot vector has a 1 for the selected category and 0 elsewhere. The model produces probability vectors \(p^{(c)}\), \(p^{(g)}\), and \(p^{(t)}\), each summing to one. The training objective is

$$\mathcal{L} = -\frac{1}{B}\sum_{b=1}^{B}\sum_j y^{(c)}_{bj}\log p^{(c)}_{bj}
-\frac{1}{B}\sum_{b=1}^{B}\sum_j y^{(g)}_{bj}\log p^{(g)}_{bj}
+\frac{1}{2}\frac{1}{3B}\sum_{b=1}^{B}\sum_{j=1}^{3}\left(p^{(t)}_{bj}-y^{(t)}_{bj}\right)^2.$$

The first two terms are categorical cross-entropy losses; the third is mean squared error for the timing distribution. \(B\) is the batch size, and \(j\) indexes categories. The timing scores are made nonnegative from future profit margin and unused berth capacity, then normalized to sum to one. The implementation updates only the three output heads using gradient descent; the encoder stays fixed.

### Solution sequence

1. Define the 24 features and create synthetic monthly observations.
2. Scale features and form overlapping 120-month input windows with 36-month future windows.
3. Apply input projection, positional encoding, multi-head self-attention, and mean pooling.
4. Train the capacity, technology, and timing output heads against rule-generated targets.
5. Inspect probabilities, sample predictions, and attention weights, while keeping the data and model limitations in view.

With \(N\) months, input length \(L\), and forecast horizon \(H\), the number of windows is \(\max(0,N-L-H+1)\). Attention computes pairwise interactions between all \(L\) positions, so its time and attention-memory costs scale approximately as \(O(L^2)\) per head and layer. This quadratic dependence is one reason long histories can be computationally demanding.

In [ ]:
from typing import List, Optional, Tuple

import math
import random
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Fixed seeds make the synthetic example and its initialized weights reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported for Transformer-based capacity planning.")

In [ ]:
@dataclass
class PortDataProcessor:
    """Generate synthetic port data and construct supervised time-series windows."""

    def __init__(self):
        self.feature_columns = [
            "monthly_teu", "berth_utilization", "gate_throughput", "dwell_time",
            "crane_productivity", "yard_density", "fuel_costs", "labor_costs",
            "gdp_growth", "trade_volume", "competitor_capacity", "shipping_rates",
            "seasonal_factor", "weather_impact", "labor_disputes", "infrastructure_age",
            "technology_index", "automation_level", "maintenance_costs",
            "customer_satisfaction", "environmental_compliance", "regulatory_changes",
            "market_share", "profit_margin",
        ]
        self.capacity_decisions = [
            "no_expansion", "minor_expansion", "major_expansion", "comprehensive"
        ]
        self.technology_decisions = [
            "current_tech", "semi_automation", "full_automation"
        ]
        self.feature_means: Optional[np.ndarray] = None
        self.feature_stds: Optional[np.ndarray] = None

    def generate_synthetic_data(self, num_months: int = 180) -> pd.DataFrame:
        """Generate reproducible synthetic monthly observations for demonstration."""
        rng = np.random.default_rng(42)
        base_values = {
            "monthly_teu": 250000,
            "berth_utilization": 0.75,
            "gate_throughput": 150,
            "dwell_time": 3.5,
            "crane_productivity": 25,
            "yard_density": 0.65,
            "fuel_costs": 100,
            "labor_costs": 120,
            "gdp_growth": 0.03,
            "trade_volume": 1.0,
            "competitor_capacity": 5.0,
            "shipping_rates": 1500,
            "seasonal_factor": 1.0,
            "weather_impact": 0.0,
            "labor_disputes": 0.0,
            "infrastructure_age": 10,
            "technology_index": 0.6,
            "automation_level": 0.3,
            "maintenance_costs": 80,
            "customer_satisfaction": 0.8,
            "environmental_compliance": 0.9,
            "regulatory_changes": 0.1,
            "market_share": 0.15,
            "profit_margin": 0.08,
        }

        rows = []
        bounded_features = {
            "berth_utilization", "yard_density", "technology_index",
            "automation_level", "customer_satisfaction", "environmental_compliance",
        }
        nonnegative_features = {"labor_disputes", "weather_impact", "regulatory_changes"}

        for month in range(num_months):
            trend_factor = 1 + 0.005 * month
            seasonal_factor = 1 + 0.1 * np.sin(2 * np.pi * month / 12)
            row = {}

            for feature, base_value in base_values.items():
                noise = rng.normal(0, 0.05)
                value = base_value * trend_factor * seasonal_factor * (1 + noise)

                if feature in bounded_features:
                    value = np.clip(value, 0, 1)
                elif feature in nonnegative_features:
                    value = max(0, value)

                row[feature] = value

            rows.append(row)

        return pd.DataFrame(rows)

    def fit_scaler(self, data: pd.DataFrame) -> None:
        """Store feature-wise means and standard deviations for input scaling."""
        values = data[self.feature_columns].to_numpy(dtype=float)
        self.feature_means = values.mean(axis=0)
        self.feature_stds = values.std(axis=0)
        self.feature_stds[self.feature_stds == 0] = 1.0

    def create_sequences(
        self,
        data: pd.DataFrame,
        seq_length: int = 120,
        forecast_horizon: int = 36,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Build overlapping input windows and targets from their future windows."""
        if self.feature_means is None or self.feature_stds is None:
            raise RuntimeError("Call fit_scaler before creating input sequences.")
        if seq_length <= 0 or forecast_horizon <= 0:
            raise ValueError("Sequence length and forecast horizon must be positive.")
        if len(data) < seq_length + forecast_horizon:
            raise ValueError(
                f"Insufficient data: need at least {seq_length + forecast_horizon} months."
            )

        raw_values = data[self.feature_columns].to_numpy(dtype=float)
        scaled_values = (raw_values - self.feature_means) / self.feature_stds
        sequences = []
        targets = []

        # The final valid window is included, hence the +1 in the range endpoint.
        for start in range(len(data) - seq_length - forecast_horizon + 1):
            input_end = start + seq_length
            future_end = input_end + forecast_horizon
            sequences.append(scaled_values[start:input_end])
            future_data = data.iloc[input_end:future_end]
            targets.append(self._extract_expansion_targets(future_data))

        return np.asarray(sequences), np.asarray(targets)

    def _extract_expansion_targets(self, future_data: pd.DataFrame) -> np.ndarray:
        """Apply transparent rules to construct capacity, technology, and timing targets."""
        max_throughput = future_data["monthly_teu"].max()
        capacity_stress = max_throughput / 267000

        if capacity_stress > 0.95:
            capacity_target = [0, 0, 0, 1]
        elif capacity_stress > 0.85:
            capacity_target = [0, 0, 1, 0]
        elif capacity_stress > 0.75:
            capacity_target = [0, 1, 0, 0]
        else:
            capacity_target = [1, 0, 0, 0]

        efficiency_trend = np.gradient(
            future_data["crane_productivity"].to_numpy()
        ).mean()
        automation_trend = np.gradient(
            future_data["automation_level"].to_numpy()
        ).mean()

        if efficiency_trend < -0.5 or automation_trend < -0.01:
            technology_target = [0, 0, 1]
        elif efficiency_trend < -0.1 or automation_trend < 0:
            technology_target = [0, 1, 0]
        else:
            technology_target = [1, 0, 0]

        # The 36-month forecast supports three annual timing categories.
        timing_scores = []
        for year in range(3):
            year_data = future_data.iloc[year * 12:(year + 1) * 12]
            mean_profit = year_data["profit_margin"].mean()
            unused_berth_capacity = 1 - year_data["berth_utilization"].mean()
            timing_scores.append(max(0, mean_profit) * max(0, unused_berth_capacity))

        timing_scores = np.asarray(timing_scores, dtype=float)
        score_total = timing_scores.sum()
        if score_total > 0:
            timing_scores /= score_total
        else:
            timing_scores[:] = 1 / len(timing_scores)

        return np.concatenate(
            [capacity_target, technology_target, timing_scores]
        ).astype(float)

print("Port data processor defined.")

In [ ]:
class SimpleTransformer:
    """A compact attention encoder with trainable multi-task output heads.

    The encoder weights are fixed after initialization. Gradient descent updates
    only the capacity, technology, and timing output heads.
    """

    def __init__(
        self,
        input_dim: int = 24,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 4,
        output_dim: int = 10,
        max_seq_length: int = 120,
    ):
        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")
        if num_layers <= 0 or max_seq_length <= 0:
            raise ValueError("num_layers and max_seq_length must be positive.")
        if output_dim != 10:
            raise ValueError("The output dimension must be 10: 4 + 3 + 3 categories.")

        self.input_dim = input_dim
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.output_dim = output_dim
        self.max_seq_length = max_seq_length
        rng = np.random.default_rng(42)

        self.W_input = rng.normal(0, 0.1, (input_dim, d_model))
        self.b_input = np.zeros(d_model)
        self.positional_encoding = self._create_positional_encoding(
            max_seq_length, d_model
        )

        self.attention_weights = []
        self.feed_forward_weights = []
        for _ in range(num_layers):
            self.attention_weights.append(
                {
                    "W_q": rng.normal(0, 0.1, (d_model, d_model)),
                    "W_k": rng.normal(0, 0.1, (d_model, d_model)),
                    "W_v": rng.normal(0, 0.1, (d_model, d_model)),
                    "W_o": rng.normal(0, 0.1, (d_model, d_model)),
                }
            )
            self.feed_forward_weights.append(
                rng.normal(0, 0.1, (d_model, d_model))
            )

        self.W_capacity = rng.normal(0, 0.1, (d_model, 4))
        self.b_capacity = np.zeros(4)
        self.W_technology = rng.normal(0, 0.1, (d_model, 3))
        self.b_technology = np.zeros(3)
        self.W_timing = rng.normal(0, 0.1, (d_model, 3))
        self.b_timing = np.zeros(3)
        self.training_loss: List[float] = []

    def _create_positional_encoding(
        self, max_seq_length: int, d_model: int
    ) -> np.ndarray:
        """Encode position with alternating sine and cosine coordinates."""
        positions = np.arange(max_seq_length).reshape(-1, 1)
        frequencies = np.exp(
            np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model)
        )
        encoding = np.zeros((max_seq_length, d_model))
        encoding[:, 0::2] = np.sin(positions * frequencies)
        encoding[:, 1::2] = np.cos(positions * frequencies)
        return encoding

    @staticmethod
    def _softmax(values: np.ndarray, axis: int = -1) -> np.ndarray:
        """Convert logits to stable probability vectors."""
        shifted = values - np.max(values, axis=axis, keepdims=True)
        exponentials = np.exp(shifted)
        return exponentials / np.sum(exponentials, axis=axis, keepdims=True)

    def _scaled_dot_product_attention(
        self, query: np.ndarray, key: np.ndarray, value: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Return attention-weighted values and their position weights."""
        key_dimension = query.shape[-1]
        scores = np.matmul(query, key.transpose(0, 1, 3, 2))
        scores = scores / np.sqrt(key_dimension)
        attention = self._softmax(scores, axis=-1)
        return np.matmul(attention, value), attention

    def _multi_head_attention(
        self, values: np.ndarray, layer_idx: int
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Project, split into heads, attend across months, and recombine."""
        batch_size, seq_len, _ = values.shape
        weights = self.attention_weights[layer_idx]
        head_dim = self.d_model // self.nhead

        query = np.matmul(values, weights["W_q"])
        key = np.matmul(values, weights["W_k"])
        value = np.matmul(values, weights["W_v"])

        query = query.reshape(batch_size, seq_len, self.nhead, head_dim)
        key = key.reshape(batch_size, seq_len, self.nhead, head_dim)
        value = value.reshape(batch_size, seq_len, self.nhead, head_dim)
        query = query.transpose(0, 2, 1, 3)
        key = key.transpose(0, 2, 1, 3)
        value = value.transpose(0, 2, 1, 3)

        attended, attention = self._scaled_dot_product_attention(
            query, key, value
        )
        attended = attended.transpose(0, 2, 1, 3).reshape(
            batch_size, seq_len, self.d_model
        )
        return np.matmul(attended, weights["W_o"]), attention

    def encode(self, inputs: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
        """Map input windows to pooled representations and retain attention maps."""
        if inputs.ndim != 3 or inputs.shape[2] != self.input_dim:
            raise ValueError("Inputs must have shape (batch, time, input_dim).")
        batch_size, seq_len, _ = inputs.shape
        if seq_len > self.max_seq_length:
            raise ValueError("Input sequence exceeds max_seq_length.")

        values = np.matmul(inputs, self.W_input) + self.b_input
        values = values * np.sqrt(self.d_model)
        values = values + self.positional_encoding[:seq_len][None, :, :]
        attention_history = []

        for layer_idx in range(self.num_layers):
            attention_output, attention = self._multi_head_attention(
                values, layer_idx
            )
            attention_history.append(attention)

            # Residual addition keeps the input representation available to later layers.
            values = values + attention_output
            norms = np.linalg.norm(values, axis=-1, keepdims=True)
            values = values / np.maximum(norms, 1e-6)

            feed_forward = np.maximum(
                0,
                np.matmul(values, self.feed_forward_weights[layer_idx]),
            )
            values = values + feed_forward

        return np.mean(values, axis=1), attention_history

    def _predict_from_representation(
        self, representation: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Apply the three task-specific heads to pooled sequence representations."""
        capacity = self._softmax(
            np.matmul(representation, self.W_capacity) + self.b_capacity
        )
        technology = self._softmax(
            np.matmul(representation, self.W_technology) + self.b_technology
        )
        timing = self._softmax(
            np.matmul(representation, self.W_timing) + self.b_timing
        )
        return capacity, technology, timing

    def forward(
        self, inputs: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Run the encoder and return capacity, technology, and timing probabilities."""
        representation, _ = self.encode(inputs)
        return self._predict_from_representation(representation)

    def train_step(
        self,
        representation_batch: np.ndarray,
        target_batch: np.ndarray,
        learning_rate: float = 0.001,
    ) -> float:
        """Update the output heads once using a batch of encoded windows."""
        if target_batch.shape[1] != self.output_dim:
            raise ValueError("Targets must contain 10 values per sequence.")

        capacity_target = target_batch[:, :4]
        technology_target = target_batch[:, 4:7]
        timing_target = target_batch[:, 7:10]
        capacity_pred, technology_pred, timing_pred = (
            self._predict_from_representation(representation_batch)
        )

        epsilon = 1e-8
        capacity_loss = -np.mean(
            np.sum(capacity_target * np.log(capacity_pred + epsilon), axis=1)
        )
        technology_loss = -np.mean(
            np.sum(technology_target * np.log(technology_pred + epsilon), axis=1)
        )
        timing_loss = np.mean((timing_pred - timing_target) ** 2)
        total_loss = capacity_loss + technology_loss + 0.5 * timing_loss

        batch_size = len(target_batch)
        capacity_gradient = (capacity_pred - capacity_target) / batch_size
        technology_gradient = (technology_pred - technology_target) / batch_size
        timing_probability_gradient = (
            (timing_pred - timing_target) / (3 * batch_size)
        )
        timing_gradient = timing_pred * (
            timing_probability_gradient
            - np.sum(timing_probability_gradient * timing_pred, axis=1, keepdims=True)
        )

        self.W_capacity -= learning_rate * np.matmul(
            representation_batch.T, capacity_gradient
        )
        self.b_capacity -= learning_rate * capacity_gradient.sum(axis=0)
        self.W_technology -= learning_rate * np.matmul(
            representation_batch.T, technology_gradient
        )
        self.b_technology -= learning_rate * technology_gradient.sum(axis=0)
        self.W_timing -= learning_rate * np.matmul(
            representation_batch.T, timing_gradient
        )
        self.b_timing -= learning_rate * timing_gradient.sum(axis=0)

        return float(total_loss)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        epochs: int = 50,
        batch_size: int = 16,
        learning_rate: float = 0.001,
    ) -> List[float]:
        """Encode windows once, then train output heads with mini-batch gradient descent."""
        if len(X_train) == 0 or len(X_train) != len(y_train):
            raise ValueError("Training inputs and targets must have equal nonzero length.")
        if epochs <= 0 or batch_size <= 0:
            raise ValueError("epochs and batch_size must be positive.")

        # The encoder is fixed, so reusing its representations avoids repeated attention work.
        representations, _ = self.encode(X_train)
        losses = []

        for epoch in range(epochs):
            batch_losses = []
            for start in range(0, len(X_train), batch_size):
                end = min(start + batch_size, len(X_train))
                loss = self.train_step(
                    representations[start:end],
                    y_train[start:end],
                    learning_rate=learning_rate,
                )
                batch_losses.append(loss)

            losses.append(float(np.mean(batch_losses)))
            if epoch % 10 == 0:
                print(f"Epoch {epoch + 1}, loss: {losses[-1]:.4f}")

        self.training_loss = losses
        return losses

print("Simple Transformer-style model defined.")

In [ ]:
processor = PortDataProcessor()

# The synthetic series has 180 months, matching the stated 15-year demonstration window.
historical_data = processor.generate_synthetic_data(num_months=180)
processor.fit_scaler(historical_data)

X_train, y_train = processor.create_sequences(
    historical_data,
    seq_length=120,
    forecast_horizon=36,
)

print(f"Synthetic observations: {len(historical_data)} months")
print(f"Feature count: {len(processor.feature_columns)}")
print(f"Input windows: {X_train.shape}")
print(f"Target matrix: {y_train.shape} (4 capacity, 3 technology, 3 timing)")
display(historical_data.head())

capacity_targets = np.argmax(y_train[:, :4], axis=1)
technology_targets = np.argmax(y_train[:, 4:7], axis=1)

print("\nRule-generated capacity target distribution:")
for index, decision in enumerate(processor.capacity_decisions):
    count = int(np.sum(capacity_targets == index))
    percentage = 100 * count / len(capacity_targets)
    print(f"  {decision}: {count} windows ({percentage:.1f}%)")

print("\nRule-generated technology target distribution:")
for index, decision in enumerate(processor.technology_decisions):
    count = int(np.sum(technology_targets == index))
    percentage = 100 * count / len(technology_targets)
    print(f"  {decision}: {count} windows ({percentage:.1f}%)")

assert X_train.shape[0] == y_train.shape[0]
assert X_train.shape[1:] == (120, len(processor.feature_columns))
assert y_train.shape[1] == 10
assert np.allclose(y_train[:, :4].sum(axis=1), 1.0)
assert np.allclose(y_train[:, 4:7].sum(axis=1), 1.0)
assert np.allclose(y_train[:, 7:10].sum(axis=1), 1.0)

In [ ]:
model = SimpleTransformer(
    input_dim=len(processor.feature_columns),
    d_model=128,
    nhead=8,
    num_layers=4,
    output_dim=10,
    max_seq_length=120,
)

print("Model configuration:")
print(f"  Input features: {model.input_dim}")
print(f"  Representation dimension: {model.d_model}")
print(f"  Attention heads: {model.nhead}")
print(f"  Encoder layers: {model.num_layers}")
print(f"  Output categories: {model.output_dim}")
print("  Trainable components: output heads only")

training_losses = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    learning_rate=0.001,
)

print(f"\nFinal training loss: {training_losses[-1]:.4f}")
print(f"Loss change from first to final epoch: {training_losses[0] - training_losses[-1]:.4f}")

In [ ]:
# These are in-sample predictions: the same windows were used to train the output heads.
capacity_probs, technology_probs, timing_probs = model.forward(X_train)
sample_size = min(10, len(X_train))
sample_indices = np.random.default_rng(42).choice(
    len(X_train), sample_size, replace=False
)

results = []
for sample_number, index in enumerate(sample_indices, start=1):
    predicted_capacity = int(np.argmax(capacity_probs[index]))
    predicted_technology = int(np.argmax(technology_probs[index]))
    predicted_timing = int(np.argmax(timing_probs[index]))
    actual_capacity = int(np.argmax(y_train[index, :4]))
    actual_technology = int(np.argmax(y_train[index, 4:7]))
    actual_timing = int(np.argmax(y_train[index, 7:10]))

    results.append(
        {
            "Sample": sample_number,
            "Predicted capacity": processor.capacity_decisions[predicted_capacity],
            "Rule target capacity": processor.capacity_decisions[actual_capacity],
            "Capacity score": capacity_probs[index, predicted_capacity],
            "Predicted technology": processor.technology_decisions[predicted_technology],
            "Rule target technology": processor.technology_decisions[actual_technology],
            "Technology score": technology_probs[index, predicted_technology],
            "Predicted timing category": predicted_timing + 1,
            "Rule target timing category": actual_timing + 1,
            "Timing score": timing_probs[index, predicted_timing],
        }
    )

results_df = pd.DataFrame(results)
display(results_df)

print(
    "The displayed scores are softmax probabilities from this model; "
    "they are not calibrated confidence intervals."
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Transformer-Style Port Capacity Planning Demonstration", fontsize=16)

axes[0, 0].plot(
    np.arange(1, len(training_losses) + 1),
    training_losses,
    color="navy",
    linewidth=2,
)
axes[0, 0].set_title("Output-Head Training Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss (dimensionless)")
axes[0, 0].grid(True, alpha=0.3)

mean_capacity_probs = capacity_probs.mean(axis=0)
axes[0, 1].bar(processor.capacity_decisions, mean_capacity_probs)
axes[0, 1].set_title("Mean Capacity-Action Probabilities")
axes[0, 1].set_ylabel("Mean probability")
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for index, probability in enumerate(mean_capacity_probs):
    axes[0, 1].text(index, probability + 0.01, f"{probability:.1%}", ha="center")

mean_technology_probs = technology_probs.mean(axis=0)
axes[1, 0].bar(processor.technology_decisions, mean_technology_probs)
axes[1, 0].set_title("Mean Technology-Choice Probabilities")
axes[1, 0].set_ylabel("Mean probability")
axes[1, 0].tick_params(axis="x", rotation=25)
axes[1, 0].grid(True, axis="y", alpha=0.3)
for index, probability in enumerate(mean_technology_probs):
    axes[1, 0].text(index, probability + 0.01, f"{probability:.1%}", ha="center")

mean_timing_probs = timing_probs.mean(axis=0)
timing_categories = np.arange(1, 4)
axes[1, 1].bar(timing_categories, mean_timing_probs, color="mediumpurple")
axes[1, 1].set_title("Mean Investment-Timing Probabilities")
axes[1, 1].set_xlabel("Annual category within the three-year forecast")
axes[1, 1].set_ylabel("Mean probability")
axes[1, 1].set_xticks(timing_categories)
axes[1, 1].grid(True, axis="y", alpha=0.3)
for category, probability in zip(timing_categories, mean_timing_probs):
    axes[1, 1].text(category, probability + 0.01, f"{probability:.1%}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the final layer's attention map for one sample and one head.
# The fixed encoder means these weights describe its computation, not learned port-specific importance.
_, attention_history = model.encode(X_train[:1])
final_layer_attention = attention_history[-1][0, 0]

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(final_layer_attention, aspect="auto", cmap="viridis")
ax.set_title("Final-Layer Attention Weights: Sample 1, Head 1")
ax.set_xlabel("Month position receiving information")
ax.set_ylabel("Month position producing the query")
fig.colorbar(image, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()

assert np.allclose(capacity_probs.sum(axis=1), 1.0)
assert np.allclose(technology_probs.sum(axis=1), 1.0)
assert np.allclose(timing_probs.sum(axis=1), 1.0)
assert np.allclose(final_layer_attention.sum(axis=-1), 1.0)

## Interpreting the demonstration

The probability plots summarize model outputs over the constructed windows; they do not establish which investment is economically optimal. The sample table compares predictions with rule-generated targets, not with independently recorded port decisions. Likewise, the loss curve measures fit to those synthetic targets and is not evidence of performance on unseen data.

An attention map shows the weights used by one head for one sample. Since the encoder weights are fixed random initializations, the map is a visualization of the attention calculation rather than a reliable explanation of which historical months drive a real capacity decision. Even with a trained encoder, attention weights alone should not be treated as a complete causal explanation.

The target rules are intentionally simple. Capacity stress is the maximum future monthly throughput divided by an assumed monthly capacity of 267,000 TEU. Technology targets use average changes in crane productivity and automation level. Timing scores combine future profit margin with unused berth capacity in each of the three forecast years. These definitions make the target construction inspectable, but they do not model project costs, financing, construction lead times, regulatory approval, uncertainty in demand, or interactions between expansion choices.

For decision support, the synthetic data and rule-generated targets would need to be replaced by appropriately governed operational data and independently validated outcomes. A realistic evaluation would also require a time-ordered training and test design, calibration checks, comparison with relevant baselines, and explicit economic and operational constraints. Those steps are outside this notebook's scope; the example's purpose is to make the Transformer-style sequence-processing and multi-task prediction workflow visible.